# Day 66：简历与作品集打磨 — STAR 法则简历生成器与 GitHub README 质量检查器

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段——**Day 66（`v1.0.0-day66`）**！今天我们用工程化的方式打磨求职“敲门砖”，端到端实测：
> 1. **简历条目体检（`audit_resume_bullets`）**：自动拦截无量化指标、无技术栈关键词、动词偏弱的“空洞表述”；
> 2. **STAR 法则结构化重构（`STARBullet` / `format_star_bullet`）**：把普通工作描述升级为可验证的高价值项目亮点；
> 3. **GitHub README 质量评分（`audit_github_readme`）**：围绕 6 大核心维度（架构图/Quickstart/API/评测/Demo/License）输出 0-100 分与等级；
> 4. **三岗位定制简历生成（`generate_resume_markdown`）**：面向 AI Application / Agent / RAG Engineer 自动重排技能与项目经历。

## 1. 环境初始化与加载 `portfolio_builder` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day66_dir = repo_root / "project_iterations" / "day_66"
if str(day66_dir) not in sys.path:
    sys.path.insert(0, str(day66_dir))

from zhishu_agent.portfolio_builder import (
    STARBullet,
    audit_github_readme,
    audit_resume_bullets,
    format_star_bullet,
    generate_resume_markdown,
)

print("已加载 Day 66 模块路径:", day66_dir)

已加载 Day 66 模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_66


## 2. 简历初稿体检 vs STAR 法则重构对比

In [2]:
weak_bullets = [
    "参与公司智能体平台开发，使用大模型和向量数据库做了问答功能，效果不错。",
    "负责后端接口联调和日常维护，提升了系统稳定性。",
]

weak_report = audit_resume_bullets(weak_bullets)
print(f"初稿体检: 通过 {weak_report.passed_bullets}/{weak_report.total_bullets} | 平均分 {weak_report.average_score}")
for item in weak_report.items:
    print(f"  - 条目#{item.index + 1} 得分={item.score} passed={item.passed} 问题={item.issues}")

assert weak_report.flagged_bullets == 2

初稿体检: 通过 0/2 | 平均分 10.0
  - 条目#1 得分=0 passed=False 问题=['缺少量化指标（Result 无具体百分比、延迟、召回率或成本数据）', '缺少核心技术关键词（未体现具体架构、框架或算法选型）', '动词偏弱或偏被动（如“参与”、“负责”、“了解”）']
  - 条目#2 得分=20 passed=False 问题=['缺少量化指标（Result 无具体百分比、延迟、召回率或成本数据）', '缺少核心技术关键词（未体现具体架构、框架或算法选型）']


In [3]:
star_bullets = [
    STARBullet(
        situation="企业财报问答中纯向量检索对股票代码与财务术语召回率不足",
        task="主导搭建混合检索与引用溯源引擎",
        action="设计 BM25 + FAISS 向量检索 + RRF 倒数秩融合链路，并集成 Cross-Encoder Rerank 精排",
        result="Top-5 召回率由 71% 提升至 92%，P95 检索延迟控制在 165ms 以内",
        tech_stack=["RAG", "BM25", "FAISS", "RRF", "Rerank"],
        role_tags=["RAG Engineer", "AI Application Engineer"],
    ),
    STARBullet(
        situation="高风险资金划拨与对外披露操作存在大模型幻觉越权风险",
        task="构建可挂起恢复的人机协同（HITL）审批闭环",
        action="基于 FastAPI SSE 与 SQLite 状态机实现 pending->approved/rejected 幂等流转与 PII 脱敏拦截",
        result="高风险动作 100% 强制人工复核，重复审批拦截率 100%，pytest 回归通过率 100%",
        tech_stack=["FastAPI", "SSE", "HITL", "SQLite", "PII", "pytest"],
        role_tags=["Agent Engineer", "AI Application Engineer"],
    ),
]

strong_report = audit_resume_bullets(star_bullets)
print(f"STAR 重构后体检: 通过 {strong_report.passed_bullets}/{strong_report.total_bullets} | 平均分 {strong_report.average_score}")
for item in strong_report.items:
    print(f"  - 条目#{item.index + 1} 得分={item.score} 指标={item.detected_metrics} 技术词={item.detected_tech_keywords}")

assert strong_report.passed_bullets == 2

STAR 重构后体检: 通过 2/2 | 平均分 100.0
  - 条目#1 得分=100 指标=['71%', '92%', '165ms'] 技术词=['RAG', 'BM25', 'RRF', 'FAISS', 'Rerank']
  - 条目#2 得分=100 指标=['100%'] 技术词=['FastAPI', 'SSE', 'HITL', 'SQLite', 'pytest', 'PII']


## 3. GitHub README 作品集质量评分

In [4]:
poor_readme = "# AI-Agent\n这是一个大模型项目。"
poor_res = audit_github_readme(poor_readme)
print("简陋版 README:", poor_res.summary)

pro_readme = """
# 智枢企业级智能体平台 (ZhiShu-Agent)
![Demo](docs/demo.gif)

## Architecture
```mermaid
flowchart LR
  UI --> API[FastAPI SSE] --> Agent[Multi-Agent + HITL]
```

## Quickstart
```bash
pip install -r requirements.txt
pytest -v
```

## API 文档
- `POST /api/v1/capstone/chat/stream`

## Benchmarks
| 方案 | Recall@5 | P95 Latency |
| --- | --- | --- |
| Hybrid RRF | 92.0% | 165ms |

## License
MIT License
"""
pro_res = audit_github_readme(pro_readme)
print("专业版 README:", pro_res.summary)
assert poor_res.score == 0 and pro_res.score == 100

简陋版 README: README 完整度评分 0/100（等级 D），暂未达到优质作品集标准；待补全项：architecture_diagram, quickstart, api_docs, eval_benchmarks, demo_gif, license。
专业版 README: README 完整度评分 100/100（等级 A），达到投递展示标准；6 大核心维度齐全！


## 4. 生成面向不同目标岗位的定制化 Markdown 简历

In [5]:
for role in ["AI Application Engineer", "Agent Engineer", "RAG Engineer"]:
    md = generate_resume_markdown(
        candidate_name="陈默",
        target_role=role,
        bullets=star_bullets,
        github_url="https://github.com/zhishu-ai/zhishu-agent",
    )
    first_line = md.splitlines()[0]
    project_lines = [l for l in md.splitlines() if l.startswith("- 针对")]
    print(f"[{role}] 标题: {first_line}")
    print(f"  置顶项目: {project_lines[0][:62]}...")

[AI Application Engineer] 标题: # 陈默 — AI 应用工程师 (AI Application Engineer)
  置顶项目: - 针对高风险资金划拨与对外披露操作存在大模型幻觉越权风险，承担构建可挂起恢复的人机协同（HITL）审批闭环；基于 Fast...
[Agent Engineer] 标题: # 陈默 — 大模型智能体工程师 (Agent Engineer)
  置顶项目: - 针对高风险资金划拨与对外披露操作存在大模型幻觉越权风险，承担构建可挂起恢复的人机协同（HITL）审批闭环；基于 Fast...
[RAG Engineer] 标题: # 陈默 — 检索增强生成工程师 (RAG Engineer)
  置顶项目: - 针对企业财报问答中纯向量检索对股票代码与财务术语召回率不足，承担主导搭建混合检索与引用溯源引擎；设计 BM25 + FA...
